## FEATURE ENGINEERING

### 1.What Data Are You Actually Collecting?

- Your stream: (btcusdt@trade)
  - returns individual BTC trades.
  - “Someone just bought/sold Bitcoin at this price and quantity.”
   


### 2. Real Binance Trade Message



```JSON
{
  "e": "trade",
  "E": 1715235000123,
  "s": "BTCUSDT",
  "t": 123456789,
  "p": "65000.50",
  "q": "0.002",
  "b": 88,
  "a": 50,
  "T": 1715235000110,
  "m": true,
  "M": true
}
```

| Field | Meaning                                        |
| ----- | ---------------------------------------------- |
| `e`   | Event type (`trade`)                           |
| `E`   | Event time (timestamp when Binance sent event) |
| `s`   | Symbol (`BTCUSDT`)                             |
| `t`   | Trade ID                                       |
| `p`   | Price of trade                                 |
| `q`   | Quantity traded                                |
| `b`   | Buyer order ID                                 |
| `a`   | Seller order ID                                |
| `T`   | Trade execution time                           |
| `m`   | Is buyer the market maker?                     |
| `M`   | Ignore field (unused by Binance)               |


## 3. Meaning of Each Field
```JSON
"m": true
```
- true → buyer is maker → aggressive side is seller
  - Is buyer the market maker?
- false → seller is maker → aggressive side is buyer

```json
"m": true
```
means:

> buyer was waiting already seller came later and sold immediately

So:
- buyer = passive
- seller = aggressive

### 4. The MOST Important Fields

| Field          | Why Important     |
| -------------- | ----------------- |
| p (price)      | market movement   |
| q (quantity)   | trade size        |
| T (time)       | sequencing        |
| m (maker flag) | buy/sell pressure |


## 5.Raw Tick Data Structure

| timestamp | price | quantity | is_buyer |
| --------- | ----- | -------- | -------- |
| 10:00:01  | 65000 | 0.01     | 1        |
| 10:00:01  | 65001 | 0.005    | 0        |
| 10:00:02  | 65003 | 0.12     | 1        |


This is called:
- tick data
- trade-level data

## 6. Why Raw Trades Are Hard For ML


- let 100 trades/sec
  - data becomes noisy
  - irregular timing
  - difficult for ML

  So we aggregate.

____

## 7. Aggregation — MOST IMPORTANT STEP
Convert trades into:

- 1-second candles 

or
- 5-second candles

This creates structured time-series data.

___

## 8. What Is a Candle?

- A candle summarizes many trades.(OHLCV)

| Time     | Open  | High  | Low   | Close | Volume |
| -------- | ----- | ----- | ----- | ----- | ------ |
| 10:00:01 | 65000 | 65010 | 64990 | 65005 | 2.1    |


## 9. Your FIRST Proper Data Structure

| timestamp | open  | high  | low   | close | volume |
| --------- | ----- | ----- | ----- | ----- | ------ |
| 10:00:01  | 65000 | 65010 | 64990 | 65005 | 2.1    |
| 10:00:02  | 65005 | 65020 | 65000 | 65018 | 1.8    |


## 10.Now Feature Engineering Begins

- Features are:
  - mathematical descriptions of market behavior.
- You convert raw prices into signals.

___

## 11. Feature #1 — Returns

- Measures price movement.

## Return Formula

$
\text{return}_t =
\frac{\text{close}_t - \text{close}_{t-1}}
{\text{close}_{t-1}}
$

Where:

- $\text{close}_t$ = current close price
- $\text{close}_{t-1}$ = previous close price

Example:

$
\frac{65010 - 65000}{65000}
= 0.0001538
$

## Feature #2 — Rolling Mean

 Measures trend.

## Moving Average Formula

$
MA_n = \frac{1}{n}\sum_{i=1}^{n} P_i
$

Where:

- $MA_n$ = moving average over $n$ periods
- $P_i$ = price at period $i$
- $n$ = number of periods

## Feature #3 — Volatility

Measures market instability.

Very important feature.

High volatility:
- large price swings

Low volatility:
- calm market

### Formula

Standard deviation of returns:

$
\sigma = \text{std}(\text{return})
$

Where:

- $\sigma$ = volatility
- $\text{return}$ = price returns

Example:

Returns:

$
[0.01,\ -0.02,\ 0.015,\ -0.005]
$

Higher spread in returns ⇒ higher volatility.

## Feature #4 — VWAP

Professional trading feature.

VWAP = Volume Weighted Average Price

It shows the average traded price weighted by volume.

### Formula

$
VWAP =
\frac{\sum (\text{price} \times \text{volume})}
{\sum \text{volume}}
$

Where:

- $\text{price}$ = trade price
- $\text{volume}$ = trade quantity

Example:

| Price | Volume |
|---|---|
| 65000 | 2 |
| 65100 | 3 |

Calculation:

$
VWAP =
\frac{(65000 \times 2) + (65100 \times 3)}
{2 + 3}
$

$$
=
\frac{130000 + 195300}{5}
$$

$$
= 65060
$$

Higher volume trades affect VWAP more strongly.

##  Feature #5 — Buy/Sell Imbalance

## Order Imbalance Formula

Measures buying pressure vs selling pressure.

### Formula

$
\text{imbalance} =
\frac{\text{buy\_volume} - \text{sell\_volume}}
{\text{buy\_volume} + \text{sell\_volume}}
$

Where:

- $\text{buy\_volume}$ = total aggressive buy volume
- $\text{sell\_volume}$ = total aggressive sell volume

Interpretation:

- Positive value → buyers stronger
- Negative value → sellers stronger
- Near 0 → balanced market

| Value     | Meaning          |
| --------- | ---------------- |
| positive  | buyers stronger  |
| negative  | sellers stronger |
| near zero | balanced         |



___
## Final ML Dataset Structure

| timestamp | close | return | volatility | volume | imbalance | vwap_dist | label |
| --------- | ----- | ------ | ---------- | ------ | --------- | --------- | ----- |
| 10:00:01  | 65000 | 0.001  | 0.003      | 2.1    | 0.25      | 15        | 1     |


___

## What Is the Label?

The label is the answer that the machine learning model tries to predict.

Example:

Predict whether the price will go up in the next 30 seconds.

### Formula

$
\text{label} =
\begin{cases}
1, & \text{if future\_return} > 0 \\
0, & \text{otherwise}
\end{cases}
$

Meaning:

- $1$ → price goes up
- $0$ → price does not go up

## How To Store Data

**Stage 1 (Simple)**
Use:

parquet files

Structure:
```text
data/
 ├── raw_trades/
 ├── candles_1s/
 ├── features/
 ```